# Sistema de recomendação (item-item)
Este notebook constrói a matriz usuário–item binária, calcula similaridade de cosseno entre produtos e gera o top 5 de itens similares ao produto de referência **GPS Garmin Vortex Maré Drift**.

In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# Carregamento dos dados
vendas_path = '../datasets/vendas_2023_2024.csv'
if not os.path.exists(vendas_path):
    fallback_path = '../features/vendas_normalizado.csv'
    if os.path.exists(fallback_path):
        vendas_path = fallback_path
    else:
        raise FileNotFoundError('Arquivo de vendas não encontrado.')

vendas = pd.read_csv(vendas_path)
produtos = pd.read_csv('../datasets/produtos_raw.csv')

# Padronização de tipos para garantir matching
vendas['id_client'] = vendas['id_client'].astype(str)
vendas['id_product'] = vendas['id_product'].astype(str)
produtos['code'] = produtos['code'].astype(str)

# Matriz Usuário × Produto binária (presença/ausência)
interacoes = vendas[['id_client', 'id_product']].drop_duplicates().copy()
interacoes['valor'] = 1

matriz_usuario_item = interacoes.pivot_table(
    index='id_client',
    columns='id_product',
    values='valor',
    aggfunc='max',
    fill_value=0
).astype(np.int8)

# Similaridade de cosseno Produto × Produto
matriz_item_usuario = matriz_usuario_item.T
sim = cosine_similarity(matriz_item_usuario.values)
sim_df = pd.DataFrame(sim, index=matriz_item_usuario.index, columns=matriz_item_usuario.index)

# Ranking dos 5 produtos mais similares ao item de referência
produto_ref_nome = 'GPS Garmin Vortex Maré Drift'
ref = produtos.loc[
    produtos['name'].str.strip().str.lower() == produto_ref_nome.strip().lower(),
    'code'
]
if ref.empty:
    raise ValueError(f"Produto de referência '{produto_ref_nome}' não encontrado na tabela de produtos.")

produto_ref_code = ref.iloc[0]

ranking = (
    sim_df.loc[produto_ref_code]
    .drop(labels=[produto_ref_code], errors='ignore')
    .sort_values(ascending=False)
    .head(5)
    .rename('similaridade')
    .reset_index()
)

ranking = ranking.rename(columns={ranking.columns[0]: 'id_product'})
ranking = ranking.merge(
    produtos[['code', 'name']],
    left_on='id_product',
    right_on='code',
    how='left'
).drop(columns=['code']).rename(columns={'name': 'produto'})

print(f'Produto de referência: {produto_ref_nome} (code={produto_ref_code})')
print(f'Matriz usuário–item: {matriz_usuario_item.shape[0]} usuários x {matriz_usuario_item.shape[1]} produtos')
ranking

Produto de referência: GPS Garmin Vortex Maré Drift (code=27)
Matriz usuário–item: 49 usuários x 150 produtos


,id_product,similaridade,produto
0,94,0.869626,Motor de Popa Volvo Magnum 276HP
1,11,0.868037,GPS Furuno Swift Leviathan Poseidon
2,35,0.853913,Radar Furuno Swift
3,115,0.850000,Cabo de Nylon Delta Force Magnum Leviathan
4,1,0.850000,Transponder AIS Maré Magnum


In [2]:
# Top 1 de similaridade para o produto de referência
ranking.head(1)

,id_product,similaridade,produto
0,94,0.869626,Motor de Popa Volvo Magnum 276HP
